# Ridge/Lasso: Shrinkage, Selection, λ-by-CV

**Week 3 · Lesson 9** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week3/lectures/09_ridge_lasso_housing.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 6.5 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** Lesson 2 priced homes from 8 hand-picked predictors.
The city assessor's file on each sale has 79. Does handing a pricing model
*everything* give better estimates, and can it still tell a client which
few features drive price?

**Why it matters:** nobody can hand-pick from 79 columns for every market,
but plain OLS given all of them overfits: it chases noise in the training
homes, and when one column is the exact sum of others (Data Prep finds
two), it has no unique answer for how to split credit among them. Lessons
7-8 built the fix (a penalty on coefficient size) and the tuning mechanics;
this notebook gives that fix room to pay off.

## 2. Data

Same dataset as Lesson 2 (sale prices for ~2,930 homes sold in Ames, Iowa),
but this time every column. `load_housing()` fills the blanks for you: in
most columns a blank means the home doesn't have that feature (no pool, no
garage, no alley), so it becomes a `"None"` category or 0, and
`LotFrontage`, the one truly unknown value, gets its neighborhood's median.
The loader's comments list every fill.

In [ ]:
from gbus564_coursework.utils.course_data import load_housing

df = load_housing()
print(f"{len(df):,} homes · {df.shape[1] - 1} predictor columns · "
      f"{df.isna().sum().sum()} missing values")
df.head()

In [ ]:
# The text columns, by how many distinct categories each has.
text_cols = df.select_dtypes(exclude="number").columns
print(f"{len(text_cols)} text columns, {df.shape[1] - 1 - len(text_cols)} numeric")
df[text_cols].nunique().sort_values(ascending=False).head(10)

### Data Prep

Two things to deal with before any model sees the data:

- **Text columns become 0/1 columns.** A regression needs numbers, so each
  of the 44 text columns gets one column per category (one-hot encoding).
  Categories seen in fewer than 10 training homes share one "infrequent"
  column, so a single clay-tile roof can't earn a coefficient of its own.
  That turns 79 predictors into 285 columns.
- **Some columns are exact sums of others** (next cell). `GrLivArea` is
  `1stFlrSF + 2ndFlrSF + LowQualFinSF` in every home, and `TotalBsmtSF` is
  the sum of its three parts. Lesson 6 showed that correlated predictors
  make coefficients unstable; an exact sum is the extreme case, where OLS
  has endlessly many equally good ways to split credit between a total and
  its parts.

As in Week 1 Lesson 8, the encoder is only *defined* here and fit in Model,
inside each pipeline, so it only ever learns from training rows. Ridge and
lasso penalize raw coefficient size (Lesson 8), so every column is also
standardized inside the pipeline. The 80/20 train/test split happens here
too: every model in Section 3 is fit on `X_train` and scored on the same
held-out `X_test`.

In [ ]:
# Two size measures are *exactly* the sum of other columns, in every home.
pd.Series({
    "GrLivArea = 1stFlrSF + 2ndFlrSF + LowQualFinSF":
        (df["GrLivArea"] == df["1stFlrSF"] + df["2ndFlrSF"] + df["LowQualFinSF"]).mean(),
    "TotalBsmtSF = BsmtFinSF1 + BsmtFinSF2 + BsmtUnfSF":
        (df["TotalBsmtSF"] == df["BsmtFinSF1"] + df["BsmtFinSF2"] + df["BsmtUnfSF"]).mean(),
}, name="Share of homes where it holds")

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="SalePrice"), df["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train):,} homes   Test: {len(X_test):,} homes")

# Defined here, fit in Model. One 0/1 column per category of each text
# column; categories seen in fewer than 10 training homes share one
# "infrequent" column. A category the encoder never saw in training (the
# test set has a few) goes into that infrequent column too, or gets all
# zeros if its field has none. Numeric columns pass through unchanged.
encode = make_column_transformer(
    (OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist",
                   sparse_output=False),
     make_column_selector(dtype_exclude="number")),
    remainder="passthrough", verbose_feature_names_out=False,
)

## 3. Model

*Baseline first — the reference every model must beat — then the model
under study.*

All on the 80/20 split from Data Prep: two `DummyRegressor` baselines
(always predict the training mean, or the training median — Week 2
Lesson 7's metric-matched pair), plain OLS on all 285 columns, and
ridge/lasso, each wrapped in `Pipeline(encode, StandardScaler(), ...)` per
Lesson 8's recipe.

### Model Selection

Ridge and lasso each need a λ. `GridSearchCV` tries 31 values from 0.1 to
100,000 with 5-fold CV on the training data (Lesson 8's recipe), then
refits the winning λ on the full training set. It wraps the *whole*
pipeline, encoder and scaler included, so both are refit on each training
fold and a validation fold never shapes its own preprocessing. (The
one-line `RidgeCV`/`LassoCV` shortcut can't do that: in
`make_pipeline(encode, StandardScaler(), RidgeCV(...))` the encoder and
scaler are fit once on all of `X_train`, and the CV happens afterwards,
inside already-processed data.)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 31 candidate λ values from 0.1 to 100,000. scikit-learn calls λ `alpha`,
# which is why the grid goes in as "ridge__alpha" / "lasso__alpha". Lasso
# is slow to fit at tiny λ on this many columns, so its grid starts at 10.
lambdas = np.logspace(-1, 5, 31)
lasso_lambdas = lambdas[lambdas >= 10]

baseline_mean = DummyRegressor(strategy="mean").fit(X_train, y_train)
baseline_median = DummyRegressor(strategy="median").fit(X_train, y_train)
ols = make_pipeline(encode, StandardScaler(), LinearRegression()).fit(X_train, y_train)

# GridSearchCV wraps the whole pipeline, so the encoder and scaler are refit
# on every training fold, then the best λ is refit on all of X_train.
# Each fit minimizes RSS + λ·penalty to get the coefficients; CV then scores
# each λ by plain held-out RMSE (no penalty). sklearn always *maximizes* a
# score, so error metrics come negated: "neg_root_mean_squared_error" is
# −RMSE, and the highest −RMSE is the lowest RMSE.
ridge_search = GridSearchCV(
    make_pipeline(encode, StandardScaler(), Ridge()),
    {"ridge__alpha": lambdas}, cv=5, scoring="neg_root_mean_squared_error",
    n_jobs=-1,
).fit(X_train, y_train)
lasso_search = GridSearchCV(
    make_pipeline(encode, StandardScaler(), Lasso(max_iter=10_000)),
    {"lasso__alpha": lasso_lambdas}, cv=5, scoring="neg_root_mean_squared_error",
    n_jobs=-1,
).fit(X_train, y_train)
ridge, lasso = ridge_search.best_estimator_, lasso_search.best_estimator_

features = ols[:-1].get_feature_names_out()  # column names after encoding
print(f"{len(features)} columns after encoding")
print(f"CV-chosen ridge λ: {ridge[-1].alpha:,.0f}   lasso λ: {lasso[-1].alpha:,.0f}")

In [ ]:
from matplotlib.ticker import StrMethodFormatter

COLORS = {"Ridge": "#2a78d6", "Lasso": "#eb6834"}
cv_rmse = {"Ridge": -ridge_search.cv_results_["mean_test_score"],
           "Lasso": -lasso_search.cv_results_["mean_test_score"]}
grids = {"Ridge": lambdas, "Lasso": lasso_lambdas}

fig, ax = plt.subplots(figsize=(9, 5))
for name, rmse in cv_rmse.items():
    ax.plot(grids[name], rmse, color=COLORS[name], linewidth=2, label=name)
    i = rmse.argmin()
    ax.plot(grids[name][i], rmse[i], "o", color=COLORS[name], markersize=8)
ax.set_xscale("log")
ax.set_ylim(25_000, 45_000)
ax.yaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
ax.set_xlabel("λ (penalty strength, log scale)")
ax.set_ylabel("CV RMSE")
ax.set_title("CV error across λ (dot = CV best)")
ax.legend(frameon=False)
plt.tight_layout()

Both curves fall before they climb. At small λ the models are close to
OLS, which overfits 285 columns: ridge's CV error starts near \$33,100. As
λ grows, the penalty strips out that variance faster than it adds bias,
and error falls to about \$30,100 for ridge and \$29,800 for lasso, both at
λ = 1,000 (about 9% lower). Past that, shrinkage starts erasing real
effects and error climbs steeply. So CV's minimum *is* the bend: the point
where added bias starts costing more than the variance it removes.

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

models = {"Baseline (mean)": baseline_mean, "Baseline (median)": baseline_median,
          "OLS": ols, "Ridge (CV)": ridge, "Lasso (CV)": lasso}
results = pd.DataFrame({
    name: {"MAE": mean_absolute_error(y_test, m.predict(X_test)),
           "RMSE": root_mean_squared_error(y_test, m.predict(X_test)),
           "Columns used": (m[-1].coef_ != 0).sum() if hasattr(m, "steps") else 0}
    for name, m in models.items()
}).T
results.style.format("{:,.0f}")

The two baselines split the metrics exactly as Week 2 Lesson 7 predicted:
the **median** wins on MAE (\$57,223 vs. \$60,383) and the **mean** wins on
RMSE (\$80,939 vs. \$83,104), because each constant minimizes the error its
metric measures. Sale prices are right-skewed (training mean ≈ \$181k,
median ≈ \$162k), which is what pulls the two apart. So compare each
model's MAE against the median row and its RMSE against the mean row — the
toughest constant for that metric.

More columns help on their own: OLS on all 285 has a test RMSE of
\$28,980, against \$36,128 for OLS on Lesson 2's 8 predictors on this same
split. The penalty helps on top of that: ridge cuts RMSE by 2% (\$28,388)
and lasso by 3.4% (\$27,982), and lasso does it with 77 of the 285 columns.
The test-set gain is smaller than CV's 9%, since one 586-home test set is a
noisier yardstick than five folds, but both point the same way.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

Four questions: **how much does lasso simplify** the model, **which
features drive price** in what's left, **how do ridge and lasso handle**
the exact sums from Data Prep, and **what happened to Lesson 2's problem
pair**, living area and room count?

In [ ]:
from sklearn.base import clone

# Refit lasso at each λ on its grid and count the columns it keeps.
kept = pd.Series(
    [(clone(lasso).set_params(lasso__alpha=lam).fit(X_train, y_train)[-1].coef_ != 0).sum()
     for lam in lasso_lambdas],
    index=lasso_lambdas,
)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(kept.index, kept, color=COLORS["Lasso"], linewidth=2)
ax.axvline(lasso[-1].alpha, color="grey", linestyle="--", linewidth=1)
ax.text(lasso[-1].alpha, len(features), " CV best", va="top", fontsize=9)
ax.set_xscale("log")
ax.set_ylim(0, len(features))
ax.set_xlabel("λ (penalty strength, log scale)")
ax.set_ylabel("Columns with a nonzero coefficient")
ax.set_title(f"Lasso: columns kept, out of {len(features)}")
plt.tight_layout()

As λ grows, lasso zeroes out columns: 232 of 285 survive at λ = 10, 77 at
CV's pick, and none past λ ≈ 60,000. CV's 77 columns come from 51 of the
79 original predictors; the other 28, including `TotRmsAbvGrd`,
`BedroomAbvGr`, and the floor-by-floor areas, add no predictive value once
the rest are known. How far past CV's pick to push λ, trading a little
accuracy for a shorter list, is a business call, and it's this week's
assignment.

In [ ]:
coefs = pd.DataFrame({
    "OLS": ols[-1].coef_, "Ridge (CV)": ridge[-1].coef_, "Lasso (CV)": lasso[-1].coef_,
}, index=features)

# Lasso's 12 biggest effects, largest at the top.
top = coefs["Lasso (CV)"].abs().nlargest(12).index[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top, coefs.loc[top, "Lasso (CV)"], color=COLORS["Lasso"])
ax.axvline(0, color="grey", linewidth=1)
ax.xaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
ax.set_xlabel("Effect ($ of price per std. dev. of predictor)")
ax.set_title("Lasso: the 12 biggest effects")
plt.tight_layout()

What's left reads like an appraiser's checklist: size (`GrLivArea`,
finished basement, garage), quality (`OverallQual`, plus excellent
basement, kitchen, and exterior ratings), age (`YearBuilt`), and location
(three high-end neighborhoods). Living area and overall quality dominate,
as they did in Lesson 2. A dummy's effect is per standard deviation of a
0/1 column, which is hard to say out loud; Lesson 10 turns effects like
these into plain-English dollars.

In [ ]:
# The two exact sums from Data Prep: each total, then its parts.
size = ["GrLivArea", "1stFlrSF", "2ndFlrSF", "LowQualFinSF",
        "TotalBsmtSF", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF"]
coefs.loc[size].style.format("{:,.0f}")

This is where the two penalties part ways. OLS splits living area's credit
among `GrLivArea`, `1stFlrSF`, and `2ndFlrSF` more or less arbitrarily:
every split that adds up to the same predictions fits equally well, so
scikit-learn just returns one of them. Ridge spreads the credit more evenly
across all three. Lasso **picks one**: it keeps `GrLivArea` at \$23,500 and
zeroes the floor-by-floor areas, since the total already carries their
information. The basement works the same way: lasso keeps the total and
the finished area and drops the unfinished area. For a client, one
living-area number beats three that overlap.

In [ ]:
# Lesson 2's correlated pair: living area and room count.
size = ["GrLivArea", "TotRmsAbvGrd"]
coefs.loc[size].style.format("{:,.0f}")

And Lesson 2's problem pair? With 8 predictors, OLS priced
`TotRmsAbvGrd` *negatively*: room count and living area move together
(ρ ≈ 0.81), so OLS couldn't cleanly separate their effects. With all 285
columns, OLS gives room count a small *positive* \$192, essentially zero.
Ridge raises it to \$3,073 by sharing living area's credit, and lasso
zeroes it: once living area is known, room count adds nothing worth a
coefficient. Lasso doesn't fix the sign, it drops the predictor.

## 6. Recommend

**Recommendation:** price homes with the **CV-chosen lasso**. It's the most
accurate model here (test RMSE \$27,982, 3.4% better than OLS on the same
columns), and it uses 77 columns instead of 285, so the pricing story fits
on a page. Lead with living area and overall quality, then age, finished
basement, garage, and location. Leave room count and floor-by-floor areas
out of the conversation: lasso found they add no predictive value once
total living area is known.

**Caveats / next steps:** this is still 2006-2010 sales from one Iowa city
(Lesson 2's caveats carry over unchanged), and lasso's choice between a
total and its parts is a simplification, not proof that the parts don't
matter. Ridge is a reasonable alternative for a team that wants every
column to keep some weight. Lesson 10 picks up from here: turning these
standardized coefficients (and, for logistic regression, odds ratios) into
the plain-English sentences a business memo actually needs.